In [1]:
# For API calls and response parsin
import pandas as pd
import requests
from pprint import pprint

# For connecting to Supabase
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

## Testing the API Connection

In [3]:
url = "https://clinicaltrials.gov/api/v2/studies"
params={"query.cond": "non-small cell lung cancer",
        "pageSize": 1000,
        "filter.advanced": "AREA[StartDate]2022"}
response = requests.get(url, params)
study = response.json()

## Inspecting the Return Object

In [7]:
for i in study["studies"]:
    try:
        print(i['protocolSection']['armsInterventionsModule']['interventions'])
    except:
        pass

[{'type': 'DRUG', 'name': 'PF-07799933', 'description': 'Tablet', 'armGroupLabels': ['Combination dose escalation (Part 2)', 'Dose expansion (Part 3) - Tumor and mutation specific Cohort 1', 'Dose expansion (Part 3) - Tumor and mutation specific Cohort 2', 'Dose expansion (Part 3) - Tumor and mutation specific Cohort 3', 'Dose expansion (Part 3) - Tumor and mutation specific Cohort 4', 'Dose expansion (Part 3) - Tumor and mutation specific Cohort 5', 'Dose expansion (Part 3) - Tumor and mutation specific Cohort 6', 'Dose expansion (Part 3) - Tumor and mutation specific Cohort 7', 'Monotherapy dose escalation (Part 1)'], 'otherNames': ['ARRY-440']}, {'type': 'DRUG', 'name': 'binimetinib', 'description': 'Tablet', 'armGroupLabels': ['Combination dose escalation (Part 2)', 'Dose expansion (Part 3) - Tumor and mutation specific Cohort 2'], 'otherNames': ['Mektovi, PF-06811462, MEK162']}, {'type': 'BIOLOGICAL', 'name': 'cetuximab', 'description': 'Injection for intravenous use', 'armGroupLa

In [4]:
study["studies"][4]['protocolSection']['armsInterventionsModule']['armGroups'][0]

{'label': 'AG-01 treated group phase 1A',
 'type': 'EXPERIMENTAL',
 'description': 'For dose escalation, subjects will be treated with increasing doses of AG01 from 1 mg/kg to 8 mg/kg. The duration of each treatment cycle is 28 days with two infusions of AG01 every 14 days.',
 'interventionNames': ['Drug: AG-01 Compound']}

## Identifying Key Variables in the Response Object

#### Study Identifiers

- NCT ID: study['studies'][2]['protocolSection']['identificationModule']['nctId']
- Sponsor's internal trial ID: study["studies"][2]['protocolSection']['identificationModule']['orgStudyIdInfo']['id']
- Study Acronym: study['studies'][2]['protocolSection']['identificationModule']['acronym']
- Study Title: study['studies'][2]['protocolSection']['identificationModule']['briefTitle']
- Long Title: study['studies'][2]['protocolSection']['identificationModule']['officialTitle']

#### Sponsor Information

- Lead sponsor: study['studies'][2]['protocolSection']['sponsorCollaboratorsModule']['leadSponsor']
- Collaborators: study['studies'][2]['protocolSection']['sponsorCollaboratorsModule']['collaborators']

#### Study Design

- Arms: study['studies'][2]['protocolSection']['armsInterventionsModule']['armGroups']
- Drugs: study['studies'][2]['protocolSection']['armsInterventionsModule']['interventions']
- Conditions: study['studies'][2]['protocolSection']['conditionsModule']['conditions']
- Endpoints: study['studies'][2]['protocolSection']['outcomesModule']
- Line of Therapy: study['studies'][2]['protocolSection']['descriptionModule']['briefSummary'] --> can be extracted from this text
- Main investigational Drug; study['studies'][2]['protocolSection']['descriptionModule']['briefSummary'] --> can be extracted from this text

#### Study Timeline

- Study status: study['studies'][2]['protocolSection']['statusModule']['overallStatus']
- Start date: study['studies'][2]['protocolSection']['statusModule']

# Response Parser

Workshopping the response parser class to extrac the key datafields from the JSON response object. Rough sketch:

- Process each section individually
- Include error handling / validation logic
- Package results in a pandas dataframe

Next Columns to Debug:

- Arms

#### Helper Functions

In [6]:
def parse_collaborators(x):
    '''Extracts the collaborators from the collaborators portion of the protocolSection'''
    try:
        return ': '.join(a.get('name') for a in x.get("collaborators"))
    except:
        return None

#### Main Parser

In [7]:
# copying the response object to change variable names
api_response_json = study.copy()

# Flattener function
def flatten_study(study: dict) -> dict:
    '''Flattens the study dictionary of the JSON API response into a single dictionary to turn into a DataFrame'''
    id_mod = study['protocolSection'].get("identificationModule", {})
    sponsor_mod = study['protocolSection'].get("sponsorCollaboratorsModule", {})
    cond_mod = study['protocolSection'].get("conditionsModule", {})
    outcome_mod = study['protocolSection'].get("outcomesModule", {})
    desc_mod = study['protocolSection'].get("descriptionModule", {})
    status_mod = study['protocolSection'].get("statusModule", {})

    extracted_dict = {
        "nct" : id_mod.get("nctId"),
        "study_acronym": id_mod.get("acronym"),
        "study_title": id_mod.get("briefTitle"),
        "study_long_title": id_mod.get("officialTitle"),
        "lead_sponsor": sponsor_mod.get("leadSponsor", {}).get("name"),
        "collaborators:": parse_collaborators(sponsor_mod),
        "conditions": cond_mod.get("conditions"),
        "endpoints": outcome_mod.get("outcomesModule"),
        "status": status_mod.get("overallStatus"),
        "start_date": status_mod.get("statusModule")
    }
    
    return extracted_dict

# Prelim testing
results = pd.DataFrame.from_dict([flatten_study(s) for s in api_response_json['studies']])

# Connecting to Supabase

In [8]:
load_dotenv()

True

In [9]:
#client = create_engine(os.getenv("SUPABASE_PROJECT_URL"))

In [10]:
#results.to_sql(name="studies",
#               con=client,
#               if_exists="replace",
#               index=False)